In [ ]:
from diffusers import DiffusionPipeline
import torch

# text → image generation using TWO diffusion models, one after another
# Model 1 creates the rough image → Model 2 refines/improves the image → final image
# the Base and Refiner are not "two experts thinking about the prompt" like two LLM agents. 
# They are two stages of the diffusion image-generation process, 
# specialized for different parts of creating the image.

base = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
base.to("cuda")
refiner = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-refiner-1.0", text_encoder_2=base.text_encoder_2, vae=base.vae, torch_dtype=torch.float16, use_safetensors=True, variant="fp16",)
refiner.to("cuda")

# Define how many steps and what % of steps to be run on each experts (80/20) here
n_steps = 40
high_noise_frac = 0.8

prompt = "Prostate cancer show - adenocarcinoma, large cell carcinoma, squamous cell carcinoma, or normal (non-cancerous) lung tissue"

# run both experts
image = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type="latent",
).images

image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=image,
).images[0]

display(image)